# Train DPoser from scratch on the AMASS of PoseD-Flow

DPoser (Lu et al., ICCV 2025; github.com/moonbow721/DPoser-X), a score-based diffusion prior on z-scored axis-angle,
trained on **the same data as RSGM** in `rsgm_posedflow_train.ipynb`, for a fair three-way comparison in
`explore/evaluation/posedflow_vs_rsgm.ipynb`. Their released checkpoint was trained on another dump of the same AMASS
datasets, with another mixture of them (the statistics of its data in its repository differ from ours by up to 0.08
rad in the mean and 23% in the covariance), so it is not used here.

- **Data:** exactly as RSGM: `~/data/AMASS_rotations_posedflow_c10`, split `posedflow`, our 19 joints (their 21
  without the toes); train every 10th of the 18.6M poses (1.86M), valid every 100th (8,601), test untouched.
- **Model:** `hp/models/dposer.py`, the network, SDE, loss and sampler of DPoser-body rebuilt without their code
  (TimeFC 1024 × 2 blocks, sub-VP SDE β 0.1 → 20, Euler–Maruyama with 1000 steps), 7.2M parameters on 19 joints.
  The z-score normalization (`mean`, `std` per coordinate) is fitted on our training poses below and stored in the
  model's `model.json`.
- **Training:** their settings (`config/models/dposer.yaml`): RAdam, lr 2e-4, 5000 steps of warm-up then cosine, gradient
  clipping 1, EMA 0.9999, 200k steps of 5120 poses (their 1280 per GPU on 4 GPUs). That is 550 epochs over our 1.86M
  poses, and about 4 h on an M-series GPU (71 ms per step). `CONFIG = "rsgm_budget"` trains with the budget of the
  RSGM runs instead (30k steps of 4096, about 30 min) as a control.
- **Their output scaling slows the start:** DPoser divides the network output by `sigmas[int(999 t)]`, the VE noise
  levels from 50 (t = 0) down to 0.01 (t = 1), a quirk of score_sde_pytorch that it keeps for its sub-VP model. At small
  t the network must output up to 50 times the noise. In a test of 2000 steps the noise prediction was still ~0 below
  t ≈ 0.5, the validation loss 0.42 and the samples 3–4 times too wide (in the normalized space); without the scaling
  (`CONFIG = "no_sigma_scale"`, otherwise the paper's settings) the loss was 0.28 after 500 steps and the samples had
  std 0.8–1.0. Their released model shows that 200k steps get past this, but look at the checkpoints on the way (see
  below the training) before you wait for the end. `"paper"` stays the run of the comparison; `"no_sigma_scale"` is a
  control (and a fallback if the paper's run does not converge).
- **Where it differs from theirs:**
  - our axis-angle has angles in [0, π] (from our quaternions), theirs is AMASS's raw axis-angle;
  - one device with batch 5120 instead of 4 × 1280 with DDP (the same gradient, up to the order of the sums);
  - the cosine of `hp.training` decays over the steps after the warm-up, theirs over all steps times the warm-up ramp;
  - checkpoints are ranked by the validation loss (EMA weights, fixed times and noise), theirs by an MPJPE of their
    validation; the evaluation uses the EMA weights of the last step, as for RSGM.
- **Seeds:** one run per seed in `SEEDS`, each in its own folder `dposer_posedflow[_<config>]_s<seed>`. A finished run is
  not trained again.

In [2]:
import json
import sys
import time
from pathlib import Path

# the code of the checkout this notebook is in (e.g. a worktree), before an editable install of another one
SRC = next(str(p / "src") for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf

from hp.evaluation.data import load_reference
from hp.models.dposer import normalization
from hp.models.io import load_model
from hp.training import cli
from hp.training.config import load_config

assert cli.__file__.startswith(SRC), f"hp comes from {cli.__file__}: restart the kernel"

# ---- parameters ----------------------------------------------------------------------
ROTATIONS = "~/data/AMASS_rotations_posedflow_c10"
SPLIT = "posedflow"
RUN_ROOT = "~/data/experiments"
CONFIG = "paper"                            # "paper": DPoser-body; "no_sigma_scale": without their output scaling; "rsgm_budget": the RSGM budget
SEEDS = [10]                                # e.g. [10, 11, 12] for the spread between seeds
DEVICE = "mps"                              # "cpu", "mps" or "cuda:0"
RETRAIN = False                             # True: train a new run even if a finished one exists
N_CHECK = 10_000                            # samples for the quick check at the end

COMMON = ["train.valid_interval=2000", "train.checkpoint_every=10000"]
CONFIGS = {
    "paper": [],  # config/models/dposer.yaml as it is
    "no_sigma_scale": ["model.args.scale_by_sigma=false"],  # the paper's settings without their output scaling
    "rsgm_budget": ["train.max_iterations=30000", "train.batch_size=4096", "train.warmup_steps=1000",
                    "train.valid_interval=1000", "train.checkpoint_every=5000"],
}
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
name_of = lambda seed: f"dposer_posedflow{'' if CONFIG == 'paper' else '_' + CONFIG}_s{seed}"

## Normalization

The mean and the standard deviation (N − 1, as their `Posenormalizer`) of the 57 axis-angle coordinates of our training
poses. They are written to `<RUN_ROOT>/dposer_posedflow_normalization.json` and computed again only if the training
data changes (its size). They go into the model's arguments, so every run of this data normalizes the same way.

In [3]:
norm_file = expand(RUN_ROOT) / "dposer_posedflow_normalization.json"
train_frames = torch.load(expand(ROTATIONS) / "PoseDFlow_train.pt", mmap=True, weights_only=True).shape[0]
norm = json.loads(norm_file.read_text()) if norm_file.is_file() else None
if norm is None or norm["rotations"] != ROTATIONS or norm["poses"] != train_frames:
    train = load_reference("train", ROTATIONS, SPLIT)
    mean, std = normalization(train)
    norm = {"rotations": ROTATIONS, "split": SPLIT, "poses": len(train), "mean": mean, "std": std}
    norm_file.parent.mkdir(parents=True, exist_ok=True)
    norm_file.write_text(json.dumps(norm, indent=1))
    del train
mean, std = np.array(norm["mean"]), np.array(norm["std"])
print(f"{norm['poses']:,} training poses: mean in [{mean.min():.3f}, {mean.max():.3f}], std in [{std.min():.3f}, {std.max():.3f}]")

argv_of = lambda seed: ["model=dposer", f"run_root={RUN_ROOT}", f"run_name={name_of(seed)}", f"seed={seed}",
                        f"data.rotations={ROTATIONS}", f"data.split={SPLIT}", "data.load_dtype=float32",
                        f"train.device={DEVICE}", f"model.args.mean={json.dumps(norm['mean'])}",
                        f"model.args.std={json.dumps(norm['std'])}", *COMMON, *CONFIGS[CONFIG]]
cfg = load_config(argv_of(SEEDS[0]))
shown = OmegaConf.to_container(cfg)
shown["model"]["args"].update(mean="[57 numbers]", std="[57 numbers]")
print(OmegaConf.to_yaml(OmegaConf.create(shown)))

iterations, batch = cfg.train.max_iterations, cfg.train.batch_size
print(f"{iterations:,} iterations x batch {batch} = {iterations * batch / train_frames:.1f} epochs over {train_frames:,} training poses")

1,861,105 training poses: mean in [-0.779, 0.809], std in [0.043, 0.538]
run_name: dposer_posedflow_s10
run_root: ~/data/experiments
seed: 10
tensorboard: true
data:
  rotations: ~/data/AMASS_rotations_posedflow_c10
  split: posedflow
  load_dtype: float32
train:
  max_iterations: 200000
  batch_size: 5120
  lr: 0.0002
  optimizer: radam
  weight_decay: 0.0
  scheduler: cosine
  lr_step_fraction: 0.7
  lr_gamma: 0.5
  warmup_steps: 5000
  clip_grad_norm: 1.0
  ema_decay: 0.9999
  valid_interval: 2000
  checkpoint_every: 10000
  keep_last: 3
  keep_best: 3
  time_limit_minutes: null
  device: mps
  num_workers: 0
  progress: true
model:
  name: dposer
  args:
    hidden: 1024
    embed: 512
    blocks: 2
    dropout: 0.1
    beta_min: 0.1
    beta_max: 20.0
    scale_by_sigma: true
    loss_eps: 1.0e-05
    sample_steps: 1000
    sample_eps: 0.001
    mean: '[57 numbers]'
    std: '[57 numbers]'

200,000 iterations x batch 5120 = 550.2 epochs over 1,861,105 training poses


## Train

In [5]:
def finished_run(root):
    # The newest run folder below root that finished (has history.json and weights.pt), or None.
    runs = sorted(p for p in root.glob("run_[0-9][0-9][0-9][0-9]") if (p / "history.json").is_file() and (p / "weights.pt").is_file())
    return runs[-1] if runs else None


run_dirs = {}
for seed in SEEDS:
    run_dir = None if RETRAIN else finished_run(expand(RUN_ROOT) / name_of(seed))
    if run_dir is not None:
        print(f"seed {seed}: already trained, {run_dir}")
    else:
        start = time.perf_counter()
        run_dir = cli.main(argv_of(seed))
        seconds = time.perf_counter() - start
        (run_dir / "train_time.json").write_text(json.dumps({"seconds": seconds, "device": DEVICE}))
        print(f"seed {seed}: trained in {seconds / 60:.1f} min, {run_dir}")
    run_dirs[seed] = run_dir

INFO:root:Using the following datasets of AMASS: ['PoseDFlow_train'].
INFO:root:Datasets ['PoseDFlow_train'] loaded. Number of samples: 1861105
INFO:root:Using the following datasets of AMASS: ['PoseDFlow_valid'].
INFO:root:Datasets ['PoseDFlow_valid'] loaded. Number of samples: 8601


Training dposer (7,215,673 parameters) on mps; run folder /Users/evgeniifedorov/data/experiments/dposer_posedflow_s10/run_0002


 12%|█▏        | 23702/200000 [33:01<4:05:40, 11.96it/s]


KeyboardInterrupt: 

**A checkpoint on the way** (every 10k steps, in `<run folder>/checkpoint_<step>`) can be checked while the run
goes on, e.g. from a second notebook or after an interrupt: run the cell of the quick check with
`run_dirs = {10: Path("<run folder>/checkpoint_020000")}`. The normalized std of the samples should approach 1;
values of 3–4 mean the network has not yet learned the small times.

If the kernel dies or the run is stopped, go on with `cli.main([f"resume={run_dir}"])` (or
`python scripts/train_model.py resume=<run folder>` in a terminal): the run continues from its newest checkpoint.

## Loss curves

Training loss (moving average) and validation loss (EMA weights, fixed times and noise) of every seed.

In [ ]:
SMOOTH = 500  # steps of the moving average

fig, ax = plt.subplots(figsize=(10, 4))
for k, (seed, run_dir) in enumerate(run_dirs.items()):
    history = json.loads((run_dir / "history.json").read_text())
    loss = np.asarray(history["loss"], dtype=float)
    window = min(SMOOTH, len(loss))
    smooth = np.convolve(loss, np.ones(window) / window, mode="valid")
    ax.plot(np.arange(window, len(loss) + 1), smooth, color=f"C{k}", alpha=0.5, label=f"seed {seed}: training")
    ax.plot(history["valid_iteration"], history["valid_loss"], "o-", ms=3, color=f"C{k}", label=f"seed {seed}: validation")
    if history["valid_loss"]:
        best = int(np.argmin(history["valid_loss"]))
        print(f"seed {seed}: validation loss last {history['valid_loss'][-1]:.4f}, "
              f"best {history['valid_loss'][best]:.4f} at step {history['valid_iteration'][best]:,}")
ax.set_xlabel("step")
ax.set_ylabel("denoising score matching loss (mean over coordinates)")
ax.set_yscale("log")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## Quick check

The EMA weights of the last step. The samples must be valid rotations, and in the normalized space of the model they
should have mean ≈ 0 and standard deviation ≈ 1 per coordinate, like the training poses. The comparison with
PoseD-Flow and RSGM, by the harness and by the protocol of the PoseD-Flow paper, is in
`explore/evaluation/posedflow_vs_rsgm.ipynb`. DPoser has no exact log-likelihood (`has_log_prob = False`): the harness
skips it.

In [ ]:
for seed, run_dir in run_dirs.items():
    model = load_model(run_dir, device=DEVICE, weights="weights_ema")
    torch.manual_seed(seed)
    start = time.perf_counter()
    samples = model.sample(N_CHECK)
    seconds = time.perf_counter() - start
    with torch.no_grad():
        x = model.normalize(samples).cpu().double()
    samples = samples.cpu().double()
    eye = torch.eye(3, dtype=torch.float64)
    print(f"seed {seed}: {N_CHECK:,} samples in {seconds:.0f} s; max |R^T R - I| = {(samples.transpose(-1, -2) @ samples - eye).abs().max():.1e}; "
          f"normalized: |mean| <= {x.mean(0).abs().max():.3f}, std in [{x.std(0).min():.3f}, {x.std(0).max():.3f}]")